# Unlearning Sweep — Class-Wise Forgetting
### Master's Research: Machine Unlearning for Multi-Class Image Classification
**Author:** Mikołaj Hajder 264478

Each Kaggle session runs **one seed** across a subset of classes (5 per session recommended).
At the end it writes `seed_{SEED}_class_results.json` — a flat list of every completed run
that `aggregate_results.py` can combine across sessions.

**Workflow per session:**
1. Set `SEED` and `CLASSES` in §0
2. Run §1 (setup)
3. Run §2 (training) — skipped automatically if checkpoints already exist
4. Run §3 (unlearning sweep) — crash-safe; re-run after a timeout to resume
5. Run §4 (collect) — writes the per-seed JSON
6. Download `seed_{SEED}_class_results.json` and run locally:
   ```bash
   python aggregate_results.py --results-dir checkpoints/
   ```

**Kaggle session budget (CIFAR-10, T4/P100):**
- Base model training    : ~35 min
- SISA ensemble training : ~2.5 h
- Naive retrain per class: ~35 min  →  **5 classes ≈ 3 h**
- ∇τ per class           : ~5 min   →  5 classes ≈ 25 min
- SISA unlearn per class : ~10 min  →  5 classes ≈ 50 min
- **Total (5 classes)    : ~7 h  — fits comfortably in a 12 h session**

Run classes 0–4 in session A, classes 5–9 in session B (same SEED).

---
## §0  Configuration — edit `SEED` and `CLASSES` each session

In [ ]:
# ── ONE seed per session ────────────────────────────────────────────────────────
SEED    = 42                  # ← change to 43, 44, … in subsequent seeds
DATASET = 'cifar10'           # 'cifar10' | 'cifar100'

# ── Which classes to forget (split across sessions to stay within 12 h) ────────
# CIFAR-10 : 10 classes  → run [0,1,2,3,4] then [5,6,7,8,9]
# CIFAR-100: 100 classes → run a representative subset, e.g. [0,10,20,...,90]
CLASSES = [0, 1, 2, 3, 4]    # ← change to [5,6,7,8,9] for second session

# ── Which methods to run ────────────────────────────────────────────────────────
RUN_NAIVE    = True
RUN_GRAD_TAU = True
RUN_SISA     = True

# ── Paths (Kaggle defaults) ─────────────────────────────────────────────────────
CKPT_DIR = '/kaggle/working/checkpoints'
DATA_DIR = '/kaggle/working/data'
REPO_DIR = '/kaggle/working/master_thesis'
REPO_URL = 'https://github.com/okejka1/master_thesis.git'

n_methods = sum([RUN_NAIVE, RUN_GRAD_TAU, RUN_SISA])
print(f'Seed       : {SEED}')
print(f'Dataset    : {DATASET}')
print(f'Classes    : {CLASSES}')
print(f'Methods    : {n_methods}')
print(f'Total runs : {len(CLASSES) * n_methods}')

---
## §1  Setup

In [ ]:
import os, sys, json, shutil, subprocess, time
import numpy as np
import pandas as pd

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR,  exist_ok=True)

# ── Clone / update repo ─────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

subprocess.run(['git', 'log', '--oneline', '-3'], cwd=REPO_DIR)

# ── Install deps ────────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
               cwd=REPO_DIR, check=True)

# ── Verify ──────────────────────────────────────────────────────────────────────
sys.path.insert(0, REPO_DIR)
import torch
print(f'PyTorch : {torch.__version__}   CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────────────────

def is_complete(json_path: str) -> bool:
    """True only when the JSON exists AND contains status='complete'."""
    if not os.path.exists(json_path):
        return False
    try:
        with open(json_path) as f:
            return json.load(f).get('status') == 'complete'
    except Exception:
        return False


def run_script(args, complete_sentinel=None, label=''):
    """Run args as a subprocess; skip if sentinel JSON is already complete."""
    tag = f'  [{label}]' if label else ''
    if complete_sentinel and is_complete(complete_sentinel):
        print(f'{tag} SKIP — complete')
        return True
    print(f'{tag} RUN  {" ".join(str(a) for a in args)}')
    t0 = time.time()
    rc = subprocess.run([sys.executable] + [str(a) for a in args], cwd=REPO_DIR).returncode
    elapsed = time.time() - t0
    print(f'{tag} {"OK" if rc == 0 else f"ERROR rc={rc}"}  ({elapsed:.0f}s)')
    return rc == 0


# ── Directory helpers ────────────────────────────────────────────────────────────
def sdir():
    """Root dir for this seed."""
    return os.path.join(CKPT_DIR, f'seed_{SEED}')

def cdir(cls):
    """Per-class checkpoint dir inside the seed dir."""
    return os.path.join(sdir(), f'class_{cls}')

# ── Result-path helpers ──────────────────────────────────────────────────────────
def naive_json(cls):
    return os.path.join(cdir(cls), f'naive_{DATASET}_results.json')

def gt_json(cls):
    return os.path.join(cdir(cls), f'grad_tau_{DATASET}_results.json')

def sisa_json(cls):
    return os.path.join(sdir(), f'sisa_unlearn_class_{cls}.json')

print('Helpers defined.')

---
## §2  Training — base models for this seed

Skipped automatically if checkpoints already exist.
After this cell finishes **click Save Version** so checkpoints persist across sessions.

> **Note:** The same base model and SISA ensemble are reused for all class experiments.
> You only need to train once per seed — even across multiple sessions.

In [ ]:
# ── Standard ResNet-18 (used by Naive + ∇τ) ─────────────────────────────────────
os.makedirs(sdir(), exist_ok=True)
base_ckpt = os.path.join(sdir(), f'resnet18_{DATASET}_best.pth')
if not os.path.exists(base_ckpt):
    run_script(
        ['train.py',
         '--config',         f'configs/{DATASET}.yaml',
         '--data-root',      DATA_DIR,
         '--checkpoint-dir', sdir(),
         '--seed',           SEED],
        label=f'train seed={SEED}',
    )
else:
    print(f'  [train seed={SEED}] SKIP — checkpoint exists')

In [ ]:
# ── SISA ensemble ────────────────────────────────────────────────────────────────
if RUN_SISA:
    sisa_meta = os.path.join(sdir(), f'sisa_{DATASET}', 'ensemble_meta.json')
    if not os.path.exists(sisa_meta):
        run_script(
            ['train_sisa.py',
             '--config',         f'configs/{DATASET}.yaml',
             '--data-root',      DATA_DIR,
             '--checkpoint-dir', sdir(),
             '--seed',           SEED],
            label=f'sisa-train seed={SEED}',
        )
    else:
        print(f'  [sisa-train seed={SEED}] SKIP — ensemble_meta.json exists')
else:
    print('SISA training skipped (RUN_SISA=False)')

---
## §3  Class Unlearning Sweep

Loops over every class in `CLASSES`.  For each:
- **Naive** writes `class_N/naive_cifar10_results.json` in two phases:
  `training_complete` (after ~35 min retrain) then `complete` (after ~5 min MIA).
  A session crash between the two phases won't repeat the retrain.
- **∇τ** and **SISA** write their JSON in one shot (both are fast).

Re-running this cell after a timeout automatically resumes — completed runs are skipped.

> **Tip:** If the session is about to expire mid-sweep, run §4 immediately to save
> partial results before the session ends.

In [ ]:
for i, cls in enumerate(CLASSES, 1):
    cd = cdir(cls)
    os.makedirs(cd, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'  Class {cls}  ({i}/{len(CLASSES)})  seed={SEED}')
    print(f'{"="*60}')

    # Copy base checkpoint into class dir so unlearn scripts can find it
    base_src = os.path.join(sdir(), f'resnet18_{DATASET}_best.pth')
    base_dst = os.path.join(cd,     f'resnet18_{DATASET}_best.pth')
    if not os.path.exists(base_dst) and os.path.exists(base_src):
        shutil.copy(base_src, base_dst)

    # ── Naive Retrain ─────────────────────────────────────────────────────────────
    if RUN_NAIVE:
        run_script(
            ['unlearn_naive.py',
             '--config',          f'configs/{DATASET}.yaml',
             '--data-root',       DATA_DIR,
             '--checkpoint-dir',  cd,
             '--seed',            SEED,
             '--forget-strategy', 'class',
             '--forget-class',    cls],
            complete_sentinel=naive_json(cls),
            label=f'naive  class={cls}',
        )

    # ── ∇τ ────────────────────────────────────────────────────────────────────────
    if RUN_GRAD_TAU:
        run_script(
            ['unlearn_grad_tau.py',
             '--config',          f'configs/{DATASET}.yaml',
             '--data-root',       DATA_DIR,
             '--checkpoint-dir',  cd,
             '--seed',            SEED,
             '--forget-strategy', 'class',
             '--forget-class',    cls],
            complete_sentinel=gt_json(cls),
            label=f'∇τ     class={cls}',
        )

    # ── SISA ──────────────────────────────────────────────────────────────────────
    # SISA runs from sdir() (that's where the shard tree lives).
    # We copy the generic unlearn_results.json to a class-specific name.
    if RUN_SISA:
        if not is_complete(sisa_json(cls)):
            ok = run_script(
                ['unlearn_sisa.py',
                 '--config',          f'configs/{DATASET}.yaml',
                 '--data-root',       DATA_DIR,
                 '--checkpoint-dir',  sdir(),
                 '--seed',            SEED,
                 '--forget-strategy', 'class',
                 '--forget-class',    cls],
                label=f'sisa   class={cls}',
            )
            if ok:
                src = os.path.join(sdir(), f'sisa_{DATASET}', 'unlearn_results.json')
                if os.path.exists(src):
                    shutil.copy(src, sisa_json(cls))
                    print(f'  → {sisa_json(cls)}')
        else:
            print(f'  [sisa class={cls}] SKIP — complete')

print('\n✓ Sweep pass done.')

---
## §4  Collect — write `seed_{SEED}_class_results.json`

Gathers every **complete** run for this seed+class combination into one flat JSON list.
Run this after §3 finishes (or at any point to capture partial progress).
This is the file you hand to `aggregate_results.py`.

In [ ]:
records = []
missing = []

for cls in CLASSES:
    for label, path_fn in [
        ('naive_retrain', naive_json),
        ('grad_tau',      gt_json),
        ('sisa',          sisa_json),
    ]:
        fp = path_fn(cls)
        if is_complete(fp):
            with open(fp) as f:
                records.append(json.load(f))
        else:
            status = None
            if os.path.exists(fp):
                try:
                    status = json.load(open(fp)).get('status', 'unknown')
                except Exception:
                    status = 'corrupt'
            missing.append({'method': label, 'class': cls, 'status': status or 'missing'})

# Write the per-seed output file
# Include the class range in the filename so multiple sessions don't collide
cls_tag  = f'{min(CLASSES)}-{max(CLASSES)}'
out_path = os.path.join(CKPT_DIR, f'seed_{SEED}_class_{cls_tag}_results.json')
with open(out_path, 'w') as f:
    json.dump(records, f, indent=2)

print(f'Written {len(records)} complete runs → {out_path}')

if missing:
    print(f'\nIncomplete ({len(missing)} runs):')
    for m in missing:
        print(f'  class={m["class"]}  {m["method"]:<16}  status={m["status"]}')

---
## §5  Quick inline summary (optional)

Shows results for the **current session's classes only**.
Full aggregation across seeds/sessions lives in `aggregate_results.py`.

In [ ]:
if not records:
    print('No complete records — run §3 then §4 first.')
else:
    df = pd.DataFrame([
        {
            'Method':       r['method'],
            'Forget Class': r.get('forget_class', r.get('forget_strategy', '?')),
            'Forget Size':  r.get('forget_size', float('nan')),
            'Test Acc':     r['after']['test_acc'],
            'Forget Acc':   r['after']['forget_acc'],
            'Retain Acc':   r['after']['retain_acc'],
            'MIA-L %':      r['after']['mia_l'] * 100,
            'MIA-E %':      r['after']['mia_e'] * 100,
            'Time (s)':     r.get('unlearn_time_s', float('nan')),
        }
        for r in records
    ])
    print(f'seed={SEED}  classes={CLASSES}  —  {len(df)} complete runs\n')
    try:
        from IPython.display import display
        display(df.set_index(['Method', 'Forget Class']).sort_index())
    except ImportError:
        print(df.to_string())